In [24]:
# =========================
# LOAD DATA (ROBUST)
# =========================

from pathlib import Path
import pandas as pd

FILE_PATH = Path("../data/processed/eda_ppg_features_test2/features_all_test2.csv")

print("Current working dir:", Path.cwd())
print("Trying to load:", FILE_PATH)

# ---- Check if file exists ----
if not FILE_PATH.exists():
    print("\n❌ File not found!")
    print("Available CSV files instead:\n")
    
    for p in Path.cwd().glob("**/*.csv"):
        print(p)
        
else:
    print("\n✅ File found")

    # ---- Try loading ----
    try:
        df = pd.read_csv(FILE_PATH)
    except:
        print("⚠️ Trying with semicolon separator...")
        df = pd.read_csv(FILE_PATH, sep=";")

    # ---- Basic info ----
    print("\nShape:", df.shape)

    print("\nColumns:")
    print(df.columns.tolist())

    print("\nDescribe before log and clipping:")

    print("\nDescribe SCRR:")
    print(df["SC_RR"].describe())

    print("\nDescribe SC_PH:")
    print(df["SC_PH"].describe())

    print("\nDescribe EDA_slope:")
    print(df["EDA_slope"].describe())

Current working dir: c:\Users\pat19\OneDrive\Skrivbord\Thesis\CLAS_SOM_project\notebooks
Trying to load: ..\data\processed\eda_ppg_features_test2\features_all_test2.csv

✅ File found

Shape: (5780, 16)

Columns:
['HR', 'HRV_RMSSD', 'RR_diff_std', 'HRV_SDNN', 'RR_mean', 'RR_std', 'RR_min', 'RR_max', 'RR_count', 'RR_quality', 'RR_valid', 'participant', 'time_sec', 'SC_PH', 'SC_RR', 'EDA_slope']

Describe before log and clipping:

Describe SCRR:
count    5780.000000
mean        0.065037
std         0.051936
min         0.000000
25%         0.016667
50%         0.050000
75%         0.100000
max         0.300000
Name: SC_RR, dtype: float64

Describe SC_PH:
count      5780.000000
mean        127.672115
std        6148.983842
min           0.002407
25%           0.511438
50%           1.910963
75%           5.838688
max      368596.318060
Name: SC_PH, dtype: float64

Describe EDA_slope:
count    5780.000000
mean        0.000375
std         0.012617
min        -0.777949
25%        -0.000555
50

In [ ]:
import numpy as np

features = ["SC_PH", "SC_RR", "EDA_slope"]

# --- SC_PH ---
df["SC_PH"] = np.log1p(df["SC_PH"])
low, high = df["SC_PH"].quantile([0.01, 0.99])
df["SC_PH"] = df["SC_PH"].clip(low, high)

# --- SC_RR ---
low, high = df["SC_RR"].quantile([0.01, 0.99])
df["SC_RR"] = df["SC_RR"].clip(low, high)

# --- SLOPE ---
low, high = df["EDA_slope"].quantile([0.01, 0.99])
df["EDA_slope"] = df["EDA_slope"].clip(low, high)

features_df = features_df[features_df["RR_quality"] > 0.7]

print("\nDescribe AFTER log and clipping:")

print("\nDescribe SCRR:")
print(df["SC_RR"].describe())

print("\nDescribe SC_PH:")
print(df["SC_PH"].describe())

print("\nDescribe EDA_slope:")
print(df["EDA_slope"].describe())


Describe AFTER log and clipping:

Describe SCRR:
count    5780.000000
mean        0.064643
std         0.050654
min         0.000000
25%         0.016667
50%         0.050000
75%         0.100000
max         0.203500
Name: SC_RR, dtype: float64

Describe SC_PH:
count    5780.000000
mean        1.334749
std         1.166041
min         0.013653
25%         0.413062
50%         1.068484
75%         1.922596
max         5.515361
Name: SC_PH, dtype: float64

Describe EDA_slope:
count    5780.000000
mean        0.000627
std         0.004572
min        -0.017704
25%        -0.000555
50%         0.000277
75%         0.001445
max         0.020735
Name: EDA_slope, dtype: float64


In [26]:
def evaluate_eda_features(df, features, participant_col="participant"):
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    report = {}

    print("\n==============================")
    print("EDA FEATURE EVALUATION REPORT")
    print("==============================")

    # =========================
    # 1. DISTRIBUTION ANALYSIS
    # =========================
    print("\n=== DISTRIBUTIONS ===")
    for f in features:
        print(f"\nFeature: {f}")
        skew = df[f].skew()
        kurt = df[f].kurtosis()

        print(f"Skewness: {skew:.3f}")
        print(f"Kurtosis: {kurt:.3f}")

        report[f] = {
            "skew": float(skew),
            "kurtosis": float(kurt)
        }

        plt.figure()
        df[f].hist(bins=50)
        plt.title(f"{f} (Skew={skew:.2f})")
        plt.show()

    # =========================
    # 2. OUTLIER DETECTION (IQR)
    # =========================
    print("\n=== OUTLIERS (IQR) ===")
    for f in features:
        Q1 = df[f].quantile(0.25)
        Q3 = df[f].quantile(0.75)
        IQR = Q3 - Q1

        outliers = df[(df[f] < Q1 - 1.5*IQR) | (df[f] > Q3 + 1.5*IQR)]
        ratio = len(outliers) / len(df)

        print(f"{f} outlier ratio: {ratio:.3f}")
        report[f]["outlier_ratio"] = float(ratio)

    # =========================
    # 3. CORRELATION MATRIX
    # =========================
    print("\n=== CORRELATIONS ===")
    corr = df[features].corr()
    print(corr)

    plt.figure()
    plt.imshow(corr, interpolation='nearest')
    plt.colorbar()
    plt.xticks(range(len(features)), features)
    plt.yticks(range(len(features)), features)
    plt.title("Feature Correlation Matrix")
    plt.show()

    report["correlation_matrix"] = corr.to_dict()

    # =========================
    # 4. PARTICIPANT VARIABILITY
    # =========================
    print("\n=== BETWEEN-PARTICIPANT VARIATION ===")
    group_means = df.groupby(participant_col)[features].mean()

    for f in features:
        std_between = group_means[f].std()
        std_total = df[f].std()

        ratio = std_between / std_total if std_total > 0 else 0

        print(f"{f}: between/total std = {ratio:.3f}")
        report[f]["between_subject_ratio"] = float(ratio)

    # =========================
    # 5. FEATURE SIGNAL QUALITY SCORE
    # =========================
    print("\n=== FEATURE QUALITY SCORE ===")

    for f in features:
        score = 0

        # låg outlier ratio är bra
        if report[f]["outlier_ratio"] < 0.05:
            score += 1

        # rimlig skewness
        if abs(report[f]["skew"]) < 2:
            score += 1

        # variation mellan personer (bra för ML)
        if report[f]["between_subject_ratio"] > 0.3:
            score += 1

        print(f"{f} score: {score}/3")
        report[f]["quality_score"] = score

    return report

In [ ]:
features = ["SC_PH", "SC_RR", "EDA_slope"]

report = evaluate_eda_features(df, features)

In [8]:
df.describe()

,HR,HRV_RMSSD,RR_diff_std,HRV_SDNN,RR_mean,RR_std,RR_min,RR_max,RR_count,RR_quality,RR_valid,time_sec,SC_PH,SC_RR,EDA_slope
count,453.000000,453.000000,453.000000,453.000000,453.000000,453.000000,453.000000,453.000000,453.000000,453.000000,453.0,453.000000,453.000000,453.000000,453.000000
mean,79.539331,40.776142,40.753486,51.247555,759.699430,50.921874,640.055188,882.350993,79.474614,0.995614,1.0,1125.000000,1.869447,0.062068,0.000962
std,6.609851,8.450214,8.444412,13.384511,59.739882,13.298452,63.979802,63.856019,6.618718,0.010002,0.0,654.557711,1.779092,0.046869,0.010607
min,68.343380,20.662634,20.658974,23.775637,580.336538,23.643181,485.000000,695.000000,68.000000,0.920000,1.0,0.000000,0.039367,0.000000,-0.044448
25%,74.654447,34.913158,34.845179,41.639829,719.166667,41.355726,600.000000,845.000000,75.000000,1.000000,1.0,555.000000,0.597505,0.033333,-0.000690
50%,77.933372,40.684726,40.664069,49.076377,769.487179,48.799888,630.000000,890.000000,78.000000,1.000000,1.0,1125.000000,1.014062,0.050000,0.000505
75%,83.555741,46.682720,46.620754,61.318658,804.066667,60.902933,685.000000,930.000000,83.000000,1.000000,1.0,1695.000000,2.989156,0.083333,0.002754
max,103.459170,62.533775,62.494010,91.843975,877.647059,91.237738,815.000000,1050.000000,104.000000,1.000000,1.0,2250.000000,6.250276,0.183333,0.032435


In [ ]:
df[df["SC_PH"] > 10]
df[df["EDA_slope"] < -0.1]